In [ ]:
from pyspark.sql import SparkSession
from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import re
from unidecode import unidecode
from sparknlp.pretrained import PretrainedPipeline

spark = SparkSession.builder \
    .appName("ReviewPreprocess") \
    .getOrCreate()

# data read
df = spark.read.csv('top_seller_top_product_reviews.csv', header=True)

# 自定义预处理UDF
def preprocess(text):
    if text is None:
        return ''
    text = text.lower()
    text = re.sub(r'\r|\n|\t', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = unidecode(text)
    return text

preprocess_udf = udf(preprocess, StringType())

df = df.withColumn('cleaned_review', preprocess_udf(df['review_comment_message']))

# lemmatization
# for model selection, visit https://sparknlp.org/models?language=pt&task=Lemmatization
pipeline = PretrainedPipeline('lemma_bosque', lang='pt')

def lemmatize(text):
    result = pipeline.annotate(text)
    return ' '.join(result['lemmas'])

lemmatize

